# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prakritibhandari07/FlyRank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Prakritibhandari07/FlyRank-ml-internship"
REPO_DIR = "FlyRank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check repo/path"
print("Starter data found. You're ready.")

Working dir: /content/FlyRank-ml-internship
Starter data found. You're ready.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page (content_id) in the starter snapshot. The starter CSV contains one current snapshot per content page; it does not include a date/month column, so the exact calendar date range cannot be verified from this file. I will use the full warehouse release for the required month-specific checks.

In [ ]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Total rows: {df.shape[0]}")
print(f"Unique content_id values: {df['content_id'].nunique()}")
print(f"Rows match unique IDs (no duplicates): {df.shape[0] == df['content_id'].nunique()}")

Total rows: 30000
Unique content_id values: 30000
Rows match unique IDs (no duplicates): True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label:** `trend_direction` (specifically,is_declining_label = (trend_direction == "down") — this is the proxy target.

**Features:** `impressions_90d`, `days_since_last_update`, `avg_position`,
`ctr`,`sessions_90d`,
— all observable signals known before any decision was made.

**Context (not used as features):** `content_id`, `client_id` — join
keys only, used for grouping/deduplication, not predictive signal.

**Excluded:** Any product decision flags (`health_score`, `priority_score`,
`action_type`) — excluded because they aren't shipped in this dataset in
the first place, and even if they were, using them would cause circular
results (the model would just learn to copy an existing rule instead of
finding real signal).

In [ ]:
print("Columns available in dataset:")
print(list(df.columns))

Columns available in dataset:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Grain check — already confirmed in Section 1 (rows == unique content_id)

# Missing values check on key features
print("Missing values per key column:")
print(df[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "word_count"]].isnull().sum())

# Label distribution
print(f"\nDeclining rate: {(df['trend_direction'] == 'down').mean():.1%}")
print(f"\nValue counts for trend_direction:")
print(df["trend_direction"].value_counts())

Missing values per key column:
impressions_90d              0
days_since_last_update       0
avg_position                 0
ctr                          0
word_count                7699
dtype: int64

Declining rate: 54.2%

Value counts for trend_direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


Verifying the claims above with real numbers.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This starter dataset has real limits I need to keep in mind:

1. **The label is a proxy, not a future outcome.** `trend_direction` is
   calculated from the current window, not a decision point followed by
   an observed future result. A stronger version would use a
   forward-looking label (e.g., "declined over the next 30 days").

2. **This is a small anonymized slice (30,000 rows), not the full
   warehouse (79M+ rows).** Any pattern found here needs to be re-verified
   at full warehouse scale before being trusted as a general finding.

3. **No causal claims possible.** Even if a page is correctly flagged as
   declining, this data can't tell me whether refreshing it would actually
   cause a recovery — that would need a controlled experiment, which this
   dataset doesn't provide.

4. **Unbalanced history isn't directly visible in the starter slice**
   (this matters more at full warehouse scale, where different clients
   have different amounts of tracking history — something I'll need to
   check via `dim_clients.gsc_data_start` / `ga4_data_start` if I move to
   the warehouse release later).

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.